In [1]:
# Import libraries and define paths required for price simulation

import pandas as pd
import numpy as np
from pathlib import Path

PROCESSED_DATA_DIR = Path("../data/processed")

FEATURES_PATH = PROCESSED_DATA_DIR / "pricing_features.parquet"
ELASTICITY_PATH = PROCESSED_DATA_DIR / "elasticity_lookup.parquet"

In [2]:
# Load the pricing features and elasticity estimates used by the simulator

pricing_df = pd.read_parquet(FEATURES_PATH)

elasticity_df = pd.read_parquet(ELASTICITY_PATH)

print("Pricing data:", pricing_df.shape)
print("Elasticity data:", elasticity_df.shape)

print("\nMissing final elasticities:",
      elasticity_df["final_elasticity"].isna().sum())

display(elasticity_df.head())

Pricing data: (13563568, 30)
Elasticity data: (7527, 6)

Missing final elasticities: 0


,item_id,store_id,cat_id,final_elasticity,elasticity_source,r_squared
0,FOODS_1_001,CA_3,FOODS,-2.897328,category_fallback,NaN
1,FOODS_1_001,TX_1,FOODS,-2.897328,category_fallback,NaN
2,FOODS_1_002,CA_1,FOODS,-2.923871,product_store,0.076821
3,FOODS_1_002,CA_2,FOODS,-2.897328,category_fallback,NaN
4,FOODS_1_002,CA_4,FOODS,-2.897328,category_fallback,NaN


In [3]:
# Merge each product-store series with its final price elasticity estimate

simulation_df = pricing_df.merge(
    elasticity_df[
        ["item_id", "store_id", "final_elasticity", "elasticity_source"]
    ],
    on=["item_id", "store_id"],
    how="left"
)

print("Simulation data shape:", simulation_df.shape)
print("Missing elasticities:", simulation_df["final_elasticity"].isna().sum())

simulation_df[
    ["date", "item_id", "store_id", "sell_price",
     "demand", "final_elasticity", "elasticity_source"]
].head()

Simulation data shape: (13563568, 32)
Missing elasticities: 0


,date,item_id,store_id,sell_price,demand,final_elasticity,elasticity_source
0,2011-02-26,FOODS_1_001,CA_3,2.0,6,-2.897328,category_fallback
1,2011-02-27,FOODS_1_001,CA_3,2.0,6,-2.897328,category_fallback
2,2011-02-28,FOODS_1_001,CA_3,2.0,1,-2.897328,category_fallback
3,2011-03-01,FOODS_1_001,CA_3,2.0,1,-2.897328,category_fallback
4,2011-03-02,FOODS_1_001,CA_3,2.0,0,-2.897328,category_fallback


In [4]:
# Verify that all rows have the values required for price simulation

required_cols = [
    "sell_price",
    "demand",
    "final_elasticity"
]

print("Missing required values:")
print(simulation_df[required_cols].isna().sum())

print("\nElasticity sources:")
print(
    simulation_df["elasticity_source"]
    .value_counts()
)

Missing required values:
sell_price          0
demand              0
final_elasticity    0
dtype: int64

Elasticity sources:
elasticity_source
category_fallback    10848075
product_store         2715493
Name: count, dtype: int64


In [5]:
# Simulate how demand changes when a product's price is changed

def simulate_price_change(current_price, new_price, current_demand, elasticity):
    price_change_pct = (new_price - current_price) / current_price

    demand_change_pct = elasticity * price_change_pct

    predicted_demand = current_demand * (1 + demand_change_pct)
    predicted_demand = max(0, predicted_demand)

    revenue_before = current_price * current_demand
    revenue_after = new_price * predicted_demand

    return {
        "current_price": current_price,
        "new_price": new_price,
        "price_change_pct": price_change_pct * 100,
        "current_demand": current_demand,
        "predicted_demand": predicted_demand,
        "demand_change_pct": demand_change_pct * 100,
        "revenue_before": revenue_before,
        "revenue_after": revenue_after
    }

In [6]:
# Test the pricing simulator using a real product-store record from the dataset

sample = simulation_df[
    (simulation_df["demand"] > 0) &
    (simulation_df["elasticity_source"] == "product_store")
].iloc[-1]

current_price = sample["sell_price"]
new_price = current_price * 0.90

result = simulate_price_change(
    current_price=current_price,
    new_price=new_price,
    current_demand=sample["demand"],
    elasticity=sample["final_elasticity"]
)

print("Product:", sample["item_id"])
print("Store:", sample["store_id"])
print("Elasticity:", round(sample["final_elasticity"], 3))

print("\nCurrent price:", round(result["current_price"], 2))
print("New price:", round(result["new_price"], 2))
print("Price change:", round(result["price_change_pct"], 2), "%")

print("\nCurrent demand:", round(result["current_demand"], 2))
print("Predicted demand:", round(result["predicted_demand"], 2))
print("Demand change:", round(result["demand_change_pct"], 2), "%")

print("\nRevenue before:", round(result["revenue_before"], 2))
print("Revenue after:", round(result["revenue_after"], 2))

Product: HOUSEHOLD_2_505
Store: TX_2
Elasticity: -1.418

Current price: 4.97
New price: 4.47
Price change: -10.0 %

Current demand: 1
Predicted demand: 1.14
Demand change: 14.18 %

Revenue before: 4.97
Revenue after: 5.11


In [7]:
# Compare multiple price scenarios to find how price affects demand and revenue

import pandas as pd

price_changes = [-20, -15, -10, -5, 0, 5, 10, 15, 20]

simulation_results = []

for change in price_changes:
    new_price = current_price * (1 + change / 100)

    result = simulate_price_change(
        current_price=current_price,
        new_price=new_price,
        current_demand=sample["demand"],
        elasticity=sample["final_elasticity"]
    )

    simulation_results.append({
        "price_change_%": change,
        "new_price": result["new_price"],
        "predicted_demand": result["predicted_demand"],
        "predicted_revenue": result["revenue_after"]
    })

scenario_df = pd.DataFrame(simulation_results)

scenario_df.round(2)

,price_change_%,new_price,predicted_demand,predicted_revenue
0,-20,3.98,1.28,5.10
1,-15,4.22,1.21,5.12
2,-10,4.47,1.14,5.11
3,-5,4.72,1.07,5.06
4,0,4.97,1.00,4.97
5,5,5.22,0.93,4.85
6,10,5.47,0.86,4.69
7,15,5.72,0.79,4.50
8,20,5.96,0.72,4.27


In [8]:
# Select the price scenario that produces the highest predicted revenue

best_scenario = scenario_df.loc[
    scenario_df["predicted_revenue"].idxmax()
]

print("Best price change:", round(best_scenario["price_change_%"], 2), "%")
print("Recommended price:", round(best_scenario["new_price"], 2))
print("Predicted demand:", round(best_scenario["predicted_demand"], 2))
print("Predicted revenue:", round(best_scenario["predicted_revenue"], 2))

revenue_improvement = (
    (best_scenario["predicted_revenue"] - (current_price * sample["demand"]))
    / (current_price * sample["demand"])
) * 100

print("Revenue improvement:", round(revenue_improvement, 2), "%")

Best price change: -15.0 %
Recommended price: 4.22
Predicted demand: 1.21
Predicted revenue: 5.12
Revenue improvement: 3.07 %


In [9]:
# Create a reusable function that recommends a revenue-maximizing price for any product-store pair

def recommend_price(item_id, store_id, current_demand):
    product = elasticity_df[
        (elasticity_df["item_id"] == item_id) &
        (elasticity_df["store_id"] == store_id)
    ]

    if product.empty:
        return None

    elasticity = product.iloc[0]["final_elasticity"]

    latest = (
        pricing_df[
            (pricing_df["item_id"] == item_id) &
            (pricing_df["store_id"] == store_id)
        ]
        .sort_values("date")
        .iloc[-1]
    )

    current_price = latest["sell_price"]
    price_changes = [-20, -15, -10, -5, 0, 5, 10, 15, 20]

    results = []

    for change in price_changes:
        new_price = current_price * (1 + change / 100)

        simulation = simulate_price_change(
            current_price,
            new_price,
            current_demand,
            elasticity
        )

        results.append({
            "price_change_%": change,
            "new_price": new_price,
            "predicted_demand": simulation["predicted_demand"],
            "predicted_revenue": simulation["revenue_after"]
        })

    results_df = pd.DataFrame(results)

    best = results_df.loc[
        results_df["predicted_revenue"].idxmax()
    ]

    return {
        "item_id": item_id,
        "store_id": store_id,
        "current_price": round(current_price, 2),
        "recommended_price": round(best["new_price"], 2),
        "price_change_%": round(best["price_change_%"], 2),
        "current_demand": round(current_demand, 2),
        "predicted_demand": round(best["predicted_demand"], 2),
        "predicted_revenue": round(best["predicted_revenue"], 2),
        "elasticity": round(elasticity, 3)
    }

recommend_price(
    item_id="HOUSEHOLD_2_505",
    store_id="TX_2",
    current_demand=1
)

{'item_id': 'HOUSEHOLD_2_505',
 'store_id': 'TX_2',
 'current_price': np.float32(4.97),
 'recommended_price': np.float64(4.22),
 'price_change_%': np.float64(-15.0),
 'current_demand': 1,
 'predicted_demand': np.float64(1.21),
 'predicted_revenue': np.float64(5.12),
 'elasticity': np.float64(-1.418)}

In [10]:
# Save the final pricing simulation data for use in the application and deployment stage

from pathlib import Path

PROCESSED_DATA_DIR = Path("../data/processed")
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

output_path = PROCESSED_DATA_DIR / "pricing_simulation_data.parquet"

simulation_df[
    [
        "date",
        "item_id",
        "store_id",
        "cat_id",
        "sell_price",
        "demand",
        "final_elasticity",
        "elasticity_source"
    ]
].to_parquet(output_path, index=False)

print("Saved to:", output_path)
print("Rows saved:", len(simulation_df))
print("Missing elasticities:", simulation_df["final_elasticity"].isna().sum())

Saved to: ..\data\processed\pricing_simulation_data.parquet
Rows saved: 13563568
Missing elasticities: 0
